<a href="https://colab.research.google.com/github/koderlad/M507D---Methods-of-Prediction/blob/main/M507D_Week_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Importing Dependencies**

In [1]:
import pandas as pd
from sklearn.model_selection import (train_test_split, RandomizedSearchCV, GridSearchCV, cross_val_score)
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline
from sklearn.linear_model import SGDClassifier

#### **For Evaluation**

In [2]:
from sklearn.metrics import (f1_score, accuracy_score, classification_report)

## **Reading Dataset**

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/m-mahdavi/teaching/refs/heads/main/datasets/mnist.csv')

In [4]:
df.head(5)

,id,class,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,31953,5,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,34452,8,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,60897,5,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,36953,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1981,3,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## **Splitting Dataset into Train/Test.**

In [5]:
X = df.drop(['id', 'class'], axis=1)
y = df['class']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

### **Quick Pre-Processing**

In [7]:
X_train.shape

(3200, 784)

In [8]:
X_train.dropna()
X_train.shape

(3200, 784)

No Null Values. Making an assumption - "Test Dataset also has no null values".

In [9]:
y_train.value_counts()

,count
class,
1,389
7,341
3,333
8,333
6,313
2,312
0,301
4,295
9,293


Distribution is at almost similar range. For similification, skipping the data balancing technique like SMOTE for this exercise.

## **Model Training And Hyperparameter Search (Validating Design Decisions)**

In [10]:
models = {
    "SGD": SGDClassifier(random_state=1)
}

In [11]:
params = {"SGD": {
    'model__loss': ['hinge', 'log_loss', 'perceptron'],
    'model__penalty': ['elasticnet', 'l2'],
    'model__alpha': [1e-5, 1e-4, 1e-3],
    'model__max_iter': [1000, 2000],
    'model__early_stopping': [True],
    'model__validation_fraction': [0.1, 0.2],
    'model__tol': [1e-3, 1e-4, 1e-2],
}}

#### **Baseline Training (with no hyper-parameter finetuning)**

In [12]:
baseline = Pipeline(steps=[
    ('standardscaler', StandardScaler()),
    ('model', models['SGD'])
])

In [13]:
baseline.fit(X_train, y_train)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('model', SGDClassifier(random_state=1))])

In [14]:
y_pred_baseline = baseline.predict(X_test)

In [15]:
baseline_accuracy = round(accuracy_score(y_test, y_pred_baseline), 4)
print("Accuracy Score: ", baseline_accuracy)

Accuracy Score:  0.8738


In [16]:
baseline_cv = cross_val_score(baseline, X_train, y_train, cv=5, scoring="accuracy")

In [17]:
print(baseline_cv)

[0.8671875 0.8828125 0.8765625 0.8890625 0.8828125]


In [18]:
print("Baseline Accuracy (Mean) - 5 Fold: ", baseline_cv.mean())

Baseline Accuracy (Mean) - 5 Fold:  0.8796875


#### **Searching best HyperParameters**

In [19]:
search = GridSearchCV(
    estimator=baseline,
    param_grid = params['SGD'],
    scoring="accuracy",
    cv = 5,
    n_jobs=-1,
)

In [20]:
search.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('standardscaler', StandardScaler()),
                                       ('model',
                                        SGDClassifier(random_state=1))]),
             n_jobs=-1,
             param_grid={'model__alpha': [1e-05, 0.0001, 0.001],
                         'model__early_stopping': [True],
                         'model__loss': ['hinge', 'log_loss', 'perceptron'],
                         'model__max_iter': [1000, 2000],
                         'model__penalty': ['elasticnet', 'l2'],
                         'model__tol': [0.001, 0.0001, 0.01],
                         'model__validation_fraction': [0.1, 0.2]},
             scoring='accuracy')

In [21]:
print("Best Score (Accuracy): ", round(search.best_score_, 4))
print("Best Params: ", search.best_params_)

Best Score (Accuracy):  0.8825
Best Params:  {'model__alpha': 0.001, 'model__early_stopping': True, 'model__loss': 'log_loss', 'model__max_iter': 1000, 'model__penalty': 'l2', 'model__tol': 0.001, 'model__validation_fraction': 0.1}


In [22]:
beast_model = search.best_estimator_

## **Model Evaluation**

In [23]:
y_pred = beast_model.predict(X_test)

In [24]:
tuned_accuracy = round(accuracy_score(y_test, y_pred), 4)
print("Accuracy Score: ", tuned_accuracy)

Accuracy Score:  0.88


In [25]:
print(classification_report(y_test, y_pred_baseline))

              precision    recall  f1-score   support

           0       0.94      0.96      0.95        75
           1       0.91      0.92      0.91        97
           2       0.88      0.82      0.85        78
           3       0.83      0.85      0.84        84
           4       0.93      0.92      0.93        74
           5       0.78      0.84      0.81        73
           6       0.92      0.86      0.89        78
           7       0.90      0.91      0.90        85
           8       0.86      0.81      0.83        83
           9       0.81      0.86      0.83        73

    accuracy                           0.87       800
   macro avg       0.87      0.87      0.87       800
weighted avg       0.87      0.87      0.87       800



In [26]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.92      0.96      0.94        75
           1       0.90      0.94      0.92        97
           2       0.93      0.82      0.87        78
           3       0.83      0.86      0.84        84
           4       0.92      0.93      0.93        74
           5       0.83      0.86      0.85        73
           6       0.91      0.88      0.90        78
           7       0.90      0.92      0.91        85
           8       0.84      0.76      0.80        83
           9       0.83      0.86      0.85        73

    accuracy                           0.88       800
   macro avg       0.88      0.88      0.88       800
weighted avg       0.88      0.88      0.88       800



#### **Comparison**

In [27]:
print(f"Baseline Accuracy: {baseline_accuracy} / Tuned Accuracy: {tuned_accuracy}")

Baseline Accuracy: 0.8738 / Tuned Accuracy: 0.88
